In [10]:
PROJECT = "urban-access-bot"

H3_TABLE = f"{PROJECT}.bot_geo.h3_10res_london_kl"
LSOA_POSTCODE_TABLE = f"{PROJECT}.bot_geo.lsoa_postcode_map_tl"
RENTS_TABLE = f"{PROJECT}.bot_rents.monthly_rents_by_postcode_oct2024_to_sept2025_ONS_me"

OUTPUT_TABLE = f"{PROJECT}.analytics.h3_london_full_dataset"

In [11]:
def generate_sql():

    sql = f"""
    CREATE OR REPLACE TABLE {OUTPUT_TABLE} AS

    WITH

    rents_h3 AS (

        SELECT
            h3.h3_id,

            AVG(SAFE_CAST(r.Mean AS FLOAT64)) AS rent_mean_avg,
            AVG(SAFE_CAST(r.Median AS FLOAT64)) AS rent_median_avg,
            AVG(SAFE_CAST(r.Lower quartile AS FLOAT64)) AS rent_lower_q_avg,
            AVG(SAFE_CAST(r.Upper quartile AS FLOAT64)) AS rent_upper_q_avg

        FROM {RENTS_TABLE} r

        JOIN {LSOA_POSTCODE_TABLE} p
            ON LOWER(TRIM(r.Postcode District)) =
               LOWER(TRIM(p.pcd7))

        JOIN {H3_TABLE} h3
            ON p.lsoa21cd = h3.LSOA code

        GROUP BY h3.h3_id
    ),

    stats_h3 AS (

        SELECT
            h.h3_id,

            AVG(d.Disabled: activities limited a lot
                + d.Disabled: activities limited a little) AS disabled,

            AVG(a.Aged 4 and under) AS preschool_age,

            AVG(a.Aged 5 to 9
                + a.Aged 10 to 14
                + a.Aged 15 to 19) AS school_age,

            AVG(a.Aged 65 to 69
                + a.Aged 70 to 74
                + a.Aged 75 to 79
                + a.Aged 80 to 84
                + a.Aged 85 and over) AS retirement_age,

            AVG(gh.Bad health
                + gh.Very bad health) AS bad_health,

            AVG(cv.value_21) AS car_van_availability,

            AVG(hh.All Households) AS households,

            AVG(imd.Index of Multiple Deprivation IMD Rank where 1 is most deprived)
                AS deprivation,

            AVG(pop.2021) AS total_population,

            AVG(emp.All usual residents aged 16 and over in employment)
                AS employment,

            AVG(q.none) AS without_education,

            AVG(q.Level 1
                + q.Level 2
                + q.Level 3) AS preuniversity_education,

            AVG(q.Apprenticeship) AS apprentice_ship,

            AVG(q.Level 4+) AS university_education

        FROM {H3_TABLE} h

        LEFT JOIN {PROJECT}.disability_by_lsoa_and_boroughs_2021_london_datastore_kl d
            ON h.LSOA code = d.LSOA code

        LEFT JOIN {PROJECT}.five_year_age_bands_by_lsoa_and_boroughs_2021_london_datastore_kl a
            ON h.LSOA code = a.LSOA code

        LEFT JOIN {PROJECT}.general_health_by_lsoa_and_boroughs_2021_london_datastore_kl gh
            ON h.LSOA code = gh.LSOA code

        LEFT JOIN {PROJECT}.household_composition_by_car_van_availability_by_lsoa_and_boroughs_2021_london_datastore_kl cv
            ON h.LSOA code = cv.LSOA code

        LEFT JOIN {PROJECT}.household_size_by_lsoa_and_boroughs_2021_london_datastore_kl hh
            ON h.LSOA code = hh.LSOA code

        LEFT JOIN {PROJECT}.index_of_multiple_deprivation_lsoa_and_boroughs_2025_gov_uk_kl imd
            ON h.LSOA code = imd.LSOA code

        LEFT JOIN {PROJECT}.mid-year_total_population_by_boroughs_1991_to_2024_ONS_kl pop
            ON h.LSOA code = pop.LSOA code

        LEFT JOIN {PROJECT}.occupation_by_lsoa_and_boroughs_2021_london_datastore_kl emp
            ON h.LSOA code = emp.LSOA code

        LEFT JOIN {PROJECT}.qualifications_by_lsoa_and_boroughs_2021_london_datastore_kl q
            ON h.LSOA code = q.LSOA code

        GROUP BY h.h3_id
    )

    SELECT
        s.*,
        r.rent_mean_avg,
        r.rent_median_avg,
        r.rent_lower_q_avg,
        r.rent_upper_q_avg

    FROM stats_h3 s
    LEFT JOIN rents_h3 r
        ON s.h3_id = r.h3_id
    """

    return sql

In [12]:
generate_sql()

'\n    CREATE OR REPLACE TABLE urban-access-bot.analytics.h3_london_full_dataset AS\n\n    WITH\n\n    rents_h3 AS (\n\n        SELECT\n            h3.h3_id,\n\n            AVG(SAFE_CAST(r.Mean AS FLOAT64)) AS rent_mean_avg,\n            AVG(SAFE_CAST(r.Median AS FLOAT64)) AS rent_median_avg,\n            AVG(SAFE_CAST(r.Lower quartile AS FLOAT64)) AS rent_lower_q_avg,\n            AVG(SAFE_CAST(r.Upper quartile AS FLOAT64)) AS rent_upper_q_avg\n\n        FROM urban-access-bot.bot_rents.monthly_rents_by_postcode_oct2024_to_sept2025_ONS_me r\n\n        JOIN urban-access-bot.bot_geo.lsoa_postcode_map_tl p\n            ON LOWER(TRIM(r.Postcode District)) =\n               LOWER(TRIM(p.postcode))\n\n        JOIN urban-access-bot.bot_geo.h3_10res_london_kl h3\n            ON p.LSOA code = h3.LSOA code\n\n        GROUP BY h3.h3_id\n    ),\n\n    stats_h3 AS (\n\n        SELECT\n            h.h3_id,\n\n            AVG(d.Disabled: activities limited a lot\n                + d.Disabled: activit

In [ ]:
from google.cloud import bigquery

client = bigquery.Client()

query = generate_sql()

job = client.query(query)
job.result()